# **Modeling ML**

Este notebook estuda a previsão de `inflation_target` com modelos de machine learning, usando variáveis teóricas em nível, lags selecionados e diferentes combinações dessas variáveis. A comparação entre modelos é feita com métricas de erro e análise de overfitting no treino e no teste.


In [ ]:
# ! pip install lightgbm

: 

In [ ]:
# ! pip install optuna 
# Optuna is a hyperparameter optimization framework that automates the process of finding the best hyperparameters for machine learning models.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt
import optuna
import pipeline_datapreparation as DataPipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 322
np.random.seed(RANDOM_STATE)

: 

---

## **Configurações**

### **Obtenção dos dados**
Os conjuntos de treino e teste já estão divididos previamente. Ambos passam pela mesma *pipeline* com seleção de variáveis, tratamento de outliers e criação dos lags selecionados.

In [ ]:
df_train = DataPipeline.prepare_dataset(
    "dados/train.csv",
    select_features=True,
    stationary=False,
    treat_outliers=True,
    create_lags=True,
)["prepared_data"]

df_test = DataPipeline.prepare_dataset(
    "dados/test.csv",
    select_features=True,
    stationary=False,
    treat_outliers=True,
    create_lags=True,
)["prepared_data"].loc["2023-07-01":]

TARGET_COL = DataPipeline.TARGET_COL

THEORETICAL_VARIABLES = [
     "epu_pt_epu",
     "ULCIN_PT_ea-qd",
     "EXPGS_PT_ea-qd",
     "IMPGS_PT_ea-qd",
     "CCI_PT_ea-md",
     "GDP_PT_ea-qd",
     "UNETOT_PT_ea-md",
     "PPIPT_ppi",
]

print("Conjunto de Treino")
display(df_train.head())

print("Conjunto de Teste")
display(df_test.head())

print(f"Shape conjunto de treino: {df_train.shape}")
print(f"Shape conjunto de teste: {df_test.shape}")

In [ ]:
import importlib
import pipeline_datapreparation as p

importlib.reload(p)
print(p.__file__)


### **Modelos**

Modelos avaliados: **`Ridge`**, **`Random Forest`** e **`LGBMRegressor`**.

In [ ]:
models = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "RF - MSE": RandomForestRegressor(
        criterion='squared_error',
        random_state=RANDOM_STATE,
    ),
    "RF - MAE": RandomForestRegressor(
        criterion='absolute_error',
        random_state=RANDOM_STATE,
    ),
    "LGBMRegressor": LGBMRegressor(        
        random_state=RANDOM_STATE,
        verbose=-1,
    ),
}

### **Avaliação**

As métricas calculadas são **`RMSE`** e **`MAE`** em valor absoluto, e **`WAPE (%)`** em percentagem. Para verificar *overfitting*, compara-se o erro de teste com o erro de treino e considera-se que existe *overfitting* quando essa diferença percentual ultrapassa **15%** em pelo menos uma das métricas.

In [ ]:
OVERFITTING_THRESHOLD_PCT = 15.0

def relative_error_increase_pct(train_metric, test_metric):
    if pd.isna(train_metric) or train_metric == 0:
        return np.nan
    return round(((test_metric - train_metric) / train_metric) * 100, 3)

def has_overfitting(train_metrics, test_metrics, threshold_pct=OVERFITTING_THRESHOLD_PCT):
    metric_increases = [
        relative_error_increase_pct(train_metric, test_metric)
        for train_metric, test_metric in zip(train_metrics, test_metrics)
    ]
    return any(pd.notna(increase) and increase > threshold_pct for increase in metric_increases)

def regression_metrics(y_true, y_pred):
    rmse = round(np.sqrt(mean_squared_error(y_true, y_pred)), 2)
    mae = round(mean_absolute_error(y_true, y_pred), 2)
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.sum(np.abs(y_true))
    wape = np.sum(np.abs(y_true - y_pred)) / denom if denom != 0 else np.nan
    return (
        rmse,
        mae,
        round(wape, 5) if not np.isnan(wape) else np.nan,
    )

metric_cols = ["rmse", "mae", "wape"]

colors = {"Treino": "#8E9091", "Teste": "#3634bc"}

### **Ranking Models**

In [ ]:
def rank_models_by_average_test_rank(performance_df):
    ranked_df = performance_df.copy()

    ranked_df["rank_test_rmse"] = ranked_df["test_rmse"].rank(
        method="average", # empates recebem a média dos ranks possíveis
        ascending=True,
    )

    ranked_df["rank_test_mae"] = ranked_df["test_mae"].rank(
        method="average",
        ascending=True,
    )

    ranked_df["rank_test_wape"] = ranked_df["test_wape"].rank(
        method="average",
        ascending=True,
    )

    ranked_df["Overfitting?"] = ranked_df.apply(
        lambda row: has_overfitting(
            [row["train_rmse"], row["train_mae"], row["train_wape"]],
            [row["test_rmse"], row["test_mae"], row["test_wape"]],
        ),
        axis=1,
    )

    ranked_df["rank_medio"] = ranked_df[
        ["rank_test_rmse", "rank_test_mae", "rank_test_wape"]
    ].mean(axis=1)

    ranked_df = (
        ranked_df
        .sort_values(["rank_medio"])
        .reset_index(drop=True)
    )

    return ranked_df


---

## **Modelação com o desfasamento da variável alvo**

Nesta seção avalia-se um modelo base usando apenas `inflation_target` e o seu primeiro lag (`inflation_target_lag_1`). O objetivo é servir de referência para comparar o desempenho dos modelos com variáveis teóricas, em nível e com lags.

A comparação entre treino e teste é feita com `RMSE`, `MAE` e `WAPE (%)`. Considera-se *overfitting* quando o erro de teste ultrapassa em mais de `15%` o erro de treino em pelo menos uma destas métricas.



In [ ]:
# target_lag_col = f"{TARGET_COL}_lag_1"
target_lag_col = [f"{TARGET_COL}_lag_{i}" for i in range(1, 4)]


# train_lag_df = df_train[[TARGET_COL, target_lag_col]].dropna().copy()
# test_lag_df = df_test[[TARGET_COL, target_lag_col]].dropna().copy()

train_lag_df = df_train[[TARGET_COL] + target_lag_col].dropna().copy()
test_lag_df = df_test[[TARGET_COL] + target_lag_col].copy()


# Ajustamento das datas de treino e teste para garantir que não haja sobreposição de datas
common_dates = train_lag_df.index.intersection(test_lag_df.index)
test_lag_df = test_lag_df.drop(index=common_dates)

X_train = train_lag_df[target_lag_col]
y_train = train_lag_df[TARGET_COL]

X_test = test_lag_df[target_lag_col]
y_test = test_lag_df[TARGET_COL]

print(f"Train: {X_train.index.min().date()} -> {X_train.index.max().date()} | {len(X_train)} obs")
print(f"Test:  {X_test.index.min().date()} -> {X_test.index.max().date()} | {len(X_test)} obs")


In [ ]:
X_train

In [ ]:
results = []
predictions = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_rmse, train_mae, train_wape = regression_metrics(y_train, y_pred_train)
    test_rmse, test_mae, test_wape = regression_metrics(y_test, y_pred_test)

    results.append({
        "model": model_name,
        "train_rmse": train_rmse,
        "train_mae": train_mae,
        "train_wape": train_wape,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_wape": test_wape,
        "relative_error_increase_pct_rmse": relative_error_increase_pct(train_rmse, test_rmse),
        "relative_error_increase_pct_mae": relative_error_increase_pct(train_mae, test_mae),
        "relative_error_increase_pct_wape": relative_error_increase_pct(train_wape, test_wape),
        "Overfitting?": has_overfitting(
            [train_rmse, train_mae, train_wape],
            [test_rmse, test_mae, test_wape],
        ),
    })

    predictions[model_name] = pd.DataFrame(
        {"y_true": y_test, "y_pred": y_pred_test},
        index=y_test.index,
    )

metrics_df = pd.DataFrame(results).sort_values("test_rmse").reset_index(drop=True)
metrics_df = rank_models_by_average_test_rank(metrics_df)
display(metrics_df)


In [ ]:
theoretical_plot_data = []
metric_display = {"rmse": "RMSE", "mae": "MAE", "wape": "WAPE (%)"}

for _, row in metrics_df.iterrows():
    for metric in metric_cols:
        theoretical_plot_data.append({
            "model": row["model"],
            "metric": metric_display[metric],
            "dataset": "Treino",
            "value": row[f"train_{metric}"],
        })

        theoretical_plot_data.append({
            "model": row["model"],
            "metric": metric_display[metric],
            "dataset": "Teste",
            "value": row[f"test_{metric}"],
        })


theoretical_plot_df = pd.DataFrame(theoretical_plot_data)

fig, axes = plt.subplots(1, 3, figsize=(19, 6), sharex=False)

metrics_order = ["RMSE", "MAE", "WAPE (%)"]

for ax, metric in zip(axes, metrics_order):
    metric_df = theoretical_plot_df[theoretical_plot_df["metric"] == metric]
    pivot_df = metric_df.pivot(index="model", columns="dataset", values="value")
    pivot_df = pivot_df[["Treino", "Teste"]]

    pivot_df.plot(
        kind="bar",
        ax=ax,
        color=[colors["Treino"], colors["Teste"]],
        width=0.72,
        edgecolor="white",
        linewidth=0.7,
    )

    ymax = pivot_df.max().max()
    ax.set_ylim(0, ymax * 1.22)
    ax.set_title(metric, fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("")
    ax.set_ylabel("Valor", fontsize=10)
    ax.tick_params(axis="x", rotation=20, labelsize=9)
    ax.tick_params(axis="y", labelsize=9)
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)

    for container in ax.containers:
        for bar in container:
            height = bar.get_height()
            bar_color = bar.get_facecolor()
            ax.annotate(
                f"{height:.2f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8.5,
                fontweight="bold",
                color=bar_color,
            )

    ax.get_legend().remove()

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.02),
    ncol=2,
    frameon=False,
)

fig.suptitle(
    "Desfasamento da variável alvo: treino vs teste",
    fontsize=15,
    fontweight="bold",
    y=1.07,
)

fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()



In [ ]:
def plot_top_model_metrics(metrics_df, top_n=1):
    top_models = metrics_df.sort_values("test_rmse").head(top_n).copy()

    metric_labels = ["RMSE", "MAE", "WAPE (%)"]
    train_cols = ["train_rmse", "train_mae", "train_wape"]
    test_cols = ["test_rmse", "test_mae", "test_wape"]

    for _, row in top_models.iterrows():
        model_name = row["model"]
        overfitting_col = "Overfitting?" if "Overfitting?" in row.index else "overfitting?"
        overfitting = row[overfitting_col]

        train_values = [row[col] for col in train_cols]
        test_values = [row[col] for col in test_cols]

        x = np.arange(len(metric_labels))
        width = 0.35

        fig, ax = plt.subplots(figsize=(11, 6.5))

        bars_train = ax.bar(
            x - width / 2,
            train_values,
            width,
            label="Treino fitted",
            color=colors["Treino"],
            edgecolor="white",
            linewidth=0.7,
        )

        bars_test = ax.bar(
            x + width / 2,
            test_values,
            width,
            label="Teste forecast",
            color=colors["Teste"],
            edgecolor="white",
            linewidth=0.7,
        )

        ymax = max(max(train_values), max(test_values))
        ax.set_ylim(0, ymax * 1.22)

        fig.suptitle(
            f"{model_name}: treino vs teste (Overfitting? - {overfitting})",
            fontsize=14,
            fontweight="bold",
            y=0.97,
        )

        fig.legend(
            handles=[bars_train, bars_test],
            labels=["Treino fitted", "Teste forecast"],
            loc="upper center",
            bbox_to_anchor=(0.5, 0.91),
            ncol=2,
            frameon=False,
        )

        ax.set_ylabel("Valor", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(metric_labels, fontsize=10)
        ax.tick_params(axis="y", labelsize=10)
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)

        for bars in [bars_train, bars_test]:
            labels = [f"{bar.get_height():.2f}" for bar in bars]
            ax.bar_label(
                bars,
                labels=labels,
                padding=3,
                fontsize=9,
                fontweight="bold",
            )

        fig.subplots_adjust(top=0.80)
        plt.show()


plot_top_model_metrics(metrics_df, top_n=1)



---

## **Modelos com variáveis teóricas em nível**

Nesta secção usam-se apenas as variáveis teóricas selecionadas, em nível. A `pipeline` é aplicada para preparar os dados e tratar os *outliers*, sem criação de lags. Os modelos são comparados com `RMSE`, `MAE` e `WAPE`, avaliando o desempenho em treino e teste.


In [ ]:
from sklearn.base import clone

level_cols = [TARGET_COL] + THEORETICAL_VARIABLES

train_levels = df_train[level_cols].dropna().copy()
test_levels = df_test[level_cols].dropna().copy()

# Ajustamento das datas de treino e teste para garantir que não haja sobreposição de datas
common_dates = train_levels.index.intersection(test_levels.index)
test_levels = test_levels.drop(index=common_dates)

X_train = train_levels[THEORETICAL_VARIABLES]
y_train = train_levels[TARGET_COL]

X_test = test_levels[THEORETICAL_VARIABLES]
y_test = test_levels[TARGET_COL]

print(f"Treino: {X_train.index.min().date()} -> {X_train.index.max().date()} | {len(X_train)} obs")
print(f"Teste:  {X_test.index.min().date()} -> {X_test.index.max().date()} | {len(X_test)} obs")


In [ ]:
results = []
predictions = {}

for model_name, model in models.items():
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    y_pred_train = fitted_model.predict(X_train)
    y_pred_test = fitted_model.predict(X_test)

    train_rmse, train_mae, train_wape = regression_metrics(y_train, y_pred_train)
    test_rmse, test_mae, test_wape = regression_metrics(y_test, y_pred_test)

    results.append({
        "model": model_name,
        "train_rmse": train_rmse,
        "train_mae": train_mae,
        "train_wape": train_wape,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_wape": test_wape,
        "Overfitting?": has_overfitting(
            [train_rmse, train_mae, train_wape],
            [test_rmse, test_mae, test_wape],
        ),
    })

    predictions[model_name] = pd.DataFrame(
        {"y_true": y_test, "y_pred": y_pred_test},
        index=y_test.index,
    )

theoretical_metrics_df = pd.DataFrame(results).sort_values("test_rmse").reset_index(drop=True)
theoretical_metrics_df = rank_models_by_average_test_rank(theoretical_metrics_df)
display(theoretical_metrics_df)


In [ ]:
def plot_top_theoretical_model_metrics(theoretical_metrics_df, top_n=1):
    top_models = theoretical_metrics_df.sort_values("test_rmse").head(top_n).copy()

    metric_labels = ["RMSE", "MAE", "WAPE (%)"]
    train_cols = ["train_rmse", "train_mae", "train_wape"]
    test_cols = ["test_rmse", "test_mae", "test_wape"]

    for _, row in top_models.iterrows():
        model_name = row["model"]
        overfitting_col = "Overfitting?" if "Overfitting?" in row.index else "overfitting?"
        overfitting = row[overfitting_col]

        train_values = [row[col] for col in train_cols]
        test_values = [row[col] for col in test_cols]

        x = np.arange(len(metric_labels))
        width = 0.35

        fig, ax = plt.subplots(figsize=(11, 6.5))

        bars_train = ax.bar(
            x - width / 2,
            train_values,
            width,
            label="Treino fitted",
            color=colors["Treino"],
            edgecolor="white",
            linewidth=0.7,
        )

        bars_test = ax.bar(
            x + width / 2,
            test_values,
            width,
            label="Teste forecast",
            color=colors["Teste"],
            edgecolor="white",
            linewidth=0.7,
        )

        ymax = max(max(train_values), max(test_values))
        ax.set_ylim(0, ymax * 1.22)

        fig.suptitle(
            f"{model_name}: treino vs teste (Overfitting? - {overfitting})",
            fontsize=14,
            fontweight="bold",
            y=0.97,
        )

        fig.legend(
            handles=[bars_train, bars_test],
            labels=["Treino fitted", "Teste forecast"],
            loc="upper center",
            bbox_to_anchor=(0.5, 0.91),
            ncol=2,
            frameon=False,
        )

        ax.set_ylabel("Valor", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(metric_labels, fontsize=10)
        ax.tick_params(axis="y", labelsize=10)
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)

        for bars in [bars_train, bars_test]:
            labels = [f"{bar.get_height():.2f}" for bar in bars]
            ax.bar_label(
                bars,
                labels=labels,
                padding=3,
                fontsize=9,
                fontweight="bold",
            )

        fig.subplots_adjust(top=0.80)
        plt.show()


plot_top_theoretical_model_metrics(theoretical_metrics_df, top_n=1)



---

## **Modelos com variáveis teóricas em nível e os respetivos lags significativos**

Nesta fase usam-se as variáveis teóricas em nivel e, para cada uma, os respetivos lags na lista `SELECTED_LAGS`. A *pipeline* aplica o tratamento de outliers e cria os lags selecionados de forma gradual.

In [ ]:
var_lags = {'GS1_fred-md': [10],
 'GS5_fred-md': [12],
 'OILPRICEx_fred-md_Eur': [1, 2, 3, 4, 5, 6, 7, 12],
 'CUSR0000SAC_fred-md': [1, 2, 11, 12],
 'PCEPI_fred-md': [12],
 'EMPENT_PT_ea-qd': [12],
 'REER42_PT_ea-md': [6, 7, 8, 9, 10, 11, 12],
 'HICPOV_PT_ea-md': [12],
 'HICPNEF_PT_ea-md': [12],
 'HICPSV_PT_ea-md': [12],
 'HICPNG_PT_ea-md': [1, 3, 4, 12],
 'DFGDP_PT_ea-qd': [12],
 'CCONFIX_PT_ea-md': [1, 2, 11, 12],
 'epu_pt_epu': [6, 7, 8],
 'GDP_PT_ea-qd': [1, 2, 3, 4, 5, 12],
 'ULCIN_PT_ea-qd': [12],
 'EXPGS_PT_ea-qd': [12],
 'IMPGS_PT_ea-qd': [2, 3],
 'CCI_PT_ea-md': [12],
 'UNETOT_PT_ea-md': [12],
 'PPIPT_ppi': [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12]}

In [ ]:
COMPUTACIONAL_VARIABLES = [
    'GS1_fred-md', 'GS5_fred-md', 'OILPRICEx_fred-md_Eur',
    'CUSR0000SAC_fred-md', 'PCEPI_fred-md', 'EMPENT_PT_ea-qd',
    'REER42_PT_ea-md', 'HICPOV_PT_ea-md', 'HICPNEF_PT_ea-md',
    'HICPSV_PT_ea-md', 'HICPNG_PT_ea-md', 'DFGDP_PT_ea-qd',
    'CCONFIX_PT_ea-md'
]

In [ ]:
def build_lag_model(lag, models=models, include_target_lags=False, variables=THEORETICAL_VARIABLES):
    lags = list()
    for key, value in var_lags.items():
        if key in variables:
            for v in value:
                if v <= lag:
                    lags.append(f'{key}_lag_{v}')

    scores_train = {}
    scores_test = {}
    if include_target_lags:
        target_lags = ["inflation_target_lag_1"]

    # root_mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
    if include_target_lags:
        X_train = df_train[variables + lags + target_lags].dropna()
    else:
        X_train = df_train[variables + lags].dropna()
    y_train = df_train[TARGET_COL].iloc[lag:]
    
    if include_target_lags:
        X_test = df_test[variables + lags + target_lags]
    else:
        X_test = df_test[variables + lags]
    y_test = df_test[TARGET_COL]
    
    for model_name, clf in models.items():
        model_score_train = dict()
        model_score_test = dict()
        clf.fit(X_train, y_train)
    
        # Train set
        pred = clf.predict(X_train)
        model_score_train['rmse'], model_score_train['mae'], model_score_train['wape'] = regression_metrics(y_train, pred)
    
        # Test set
        pred = clf.predict(X_test)
        model_score_test['rmse'], model_score_test['mae'], model_score_test['wape'] = regression_metrics(y_test, pred)

        scores_train[model_name] = model_score_train
        scores_test[model_name] = model_score_test

    return X_test, scores_train, scores_test

In [ ]:
def evaluate_lag_performance(
    variables,
    include_target_lags=False,
    lag_range=range(1, 13),
    models=models,
):
    lag_performance = {}

    for lag in lag_range:
        X_test, scores_train, scores_test = build_lag_model(
            lag=lag,
            models=models,
            include_target_lags=include_target_lags,
            variables=variables,
        )

        lag_performance[lag] = {
            "train": scores_train,
            "test": scores_test,
        }

    lag_performance_rows = []

    for lag, lag_results in lag_performance.items():
        for model_name in lag_results["train"].keys():
            row = {
                "lag": lag,
                "model": model_name,
                "train_rmse": lag_results["train"][model_name]["rmse"],
                "train_mae": lag_results["train"][model_name]["mae"],
                "train_wape": lag_results["train"][model_name]["wape"],
                "test_rmse": lag_results["test"][model_name]["rmse"],
                "test_mae": lag_results["test"][model_name]["mae"],
                "test_wape": lag_results["test"][model_name]["wape"],
            }
            lag_performance_rows.append(row)

    lag_performance_df = pd.DataFrame(lag_performance_rows)
    lag_performance_ranked_df = rank_models_by_average_test_rank(lag_performance_df)

    return {
        "raw_results": lag_performance,
        "performance_df": lag_performance_df,
        "ranked_df": lag_performance_ranked_df,
    }



In [ ]:
results_computacional = evaluate_lag_performance(
    variables=COMPUTACIONAL_VARIABLES,
    include_target_lags=True,
)

results_theoretical = evaluate_lag_performance(
    variables=THEORETICAL_VARIABLES,
    include_target_lags=False, # O melhor modelo com as variáveis teóricas não inclui lags da variável alvo
)


In [ ]:
lag_performance_df_theoretical = results_theoretical["ranked_df"][0:5]
print("Top 5 - Teóricas:")
lag_performance_df_theoretical

In [ ]:
lag_performance_df_computacional = results_computacional["ranked_df"][0:5]
print("Top 5 - Computacionais:")    
lag_performance_df_computacional

In [ ]:
def plot_top_lag_model_metrics(lag_performance_ranked_df):
    top5_models = lag_performance_ranked_df.head(3).copy()

    metric_labels = ["RMSE", "MAE", "WAPE (%)"]
    train_cols = ["train_rmse", "train_mae", "train_wape"]
    test_cols = ["test_rmse", "test_mae", "test_wape"]

    for _, row in top5_models.iterrows():
        model_name = row["model"]
        lag = row["lag"]
        overfitting = row["Overfitting?"]

        train_values = [row[col] for col in train_cols]
        test_values = [row[col] for col in test_cols]

        x = np.arange(len(metric_labels))
        width = 0.35

        fig, ax = plt.subplots(figsize=(11, 6.5))

        bars_train = ax.bar(
            x - width / 2,
            train_values,
            width,
            label="Treino fitted",
            color=colors["Treino"],
            edgecolor="white",
            linewidth=0.7,
        )

        bars_test = ax.bar(
            x + width / 2,
            test_values,
            width,
            label="Teste forecast",
            color=colors["Teste"],
            edgecolor="white",
            linewidth=0.7,
        )

        ymax = max(max(train_values), max(test_values))
        ax.set_ylim(0, ymax * 1.22)

        fig.suptitle(
            f"{model_name} - lag {lag}: treino vs teste (Overfitting? - {overfitting})",
            fontsize=14,
            fontweight="bold",
            y=0.97,
        )

        fig.legend(
            handles=[bars_train, bars_test],
            labels=["Treino fitted", "Teste forecast"],
            loc="upper center",
            bbox_to_anchor=(0.5, 0.91),
            ncol=2,
            frameon=False,
        )

        ax.set_ylabel("Valor", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(metric_labels, fontsize=10)
        ax.tick_params(axis="y", labelsize=10)
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)

        for bars in [bars_train, bars_test]:
            labels = [f"{bar.get_height():.2f}" for bar in bars]
            ax.bar_label(
                bars,
                labels=labels,
                padding=3,
                fontsize=9,
                fontweight="bold",
            )

        fig.subplots_adjust(top=0.80)
        plt.show()


plot_top_lag_model_metrics(lag_performance_df_theoretical)



In [ ]:
print(f"Best Theoretical Model by increasing the lags number: {lag_performance_df_theoretical.loc[0, 'model']} with lag {lag_performance_df_theoretical.loc[0, 'lag']} and Overfitting? - {lag_performance_df_theoretical.loc[0, 'Overfitting?']}")
print(f"Best Computational Model by increasing the lags number: {lag_performance_df_computacional.loc[0, 'model']} with lag {lag_performance_df_computacional.loc[0, 'lag']} and Overfitting? - {lag_performance_df_computacional.loc[0, 'Overfitting?']}") 

---

## **Feature Importance of the Selected Theoretical Model**

In [ ]:
def get_lag_xy(lag, include_target_lag=True, get_features_selected=False):
    lags = []
    for key, values in var_lags.items():
        if key in THEORETICAL_VARIABLES:
            for v in values:
                if v <= lag:
                    lags.append(f"{key}_lag_{v}")

    target_lags = ["inflation_target_lag_1"] if include_target_lag else []
    feature_cols = THEORETICAL_VARIABLES + lags + target_lags
    model_cols = [TARGET_COL] + feature_cols

    train_data = df_train[model_cols].dropna().copy()
    test_data = df_test[model_cols].dropna().copy()

    X_train = train_data[feature_cols]
    y_train = train_data[TARGET_COL]

    X_test = test_data[feature_cols]
    y_test = test_data[TARGET_COL]

    if get_features_selected:
        return X_train, y_train, X_test, y_test, feature_cols
    else:
        return X_train, y_train, X_test, y_test


**LGBM Model**

In [ ]:
X_train, y_train, X_test, y_test, feature_cols = get_lag_xy(
    lag=10,
    include_target_lag=False,
    get_features_selected=True,
)

lgbm = LGBMRegressor(random_state=RANDOM_STATE, verbose=-1)
lgbm.fit(X_train, y_train)

y_pred_train = lgbm.predict(X_train)
y_pred_test = lgbm.predict(X_test)

train_rmse, train_mae, train_wape = regression_metrics(y_train, y_pred_train)
test_rmse, test_mae, test_wape = regression_metrics(y_test, y_pred_test)

print("Train RMSE:", train_rmse)
print("Train MAE:", train_mae)
print("Train WAPE (%):", train_wape)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test WAPE (%):", test_wape)

importance_df = pd.DataFrame({
    "feature": X_train.columns,
    "importance": lgbm.feature_importances_,
})

importance_plot_df = (
    importance_df.sort_values("importance", ascending=False)
    .head(10)
    .sort_values("importance", ascending=True)
)

plt.figure(figsize=(8, 7))
plt.barh(
    importance_plot_df["feature"],
    importance_plot_df["importance"],
    color=colors["Teste"],
)

plt.title("Importância das Variáveis Teóricas - LGBM (lag=10)", fontsize=12, fontweight="bold")
plt.xlabel("Importância", fontsize=10, fontweight="bold")
plt.ylabel("Variáveis")
plt.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()



---

## **Most Important Variables (Theoretical + Computational)**

In [ ]:
BEST_FEATURES = [
    "PPIPT_ppi",
    "CCI_PT_ea-md",
    "EXPGS_PT_ea-qd",
    "epu_pt_epu_lag_6",
    "PPIPT_ppi_lag_1",
    "HICPNG_PT_ea-md",
   "HICPOV_PT_ea-md",
    "HICPNEF_PT_ea-md",
    "HICPSV_PT_ea-md_lag_12",
    "HICPNEF_PT_ea-md_lag_12",
    "HICPOV_PT_ea-md_lag_12",
    "HICPNG_PT_ea-md_lag_12",
    "inflation_target_lag_1",
    "PCEPI_fred-md",
]

In [ ]:
from sklearn.base import clone

model_cols = [TARGET_COL] + BEST_FEATURES

train_best_data = df_train[model_cols].dropna().copy()
test_best_data = df_test[model_cols].dropna().copy()

common_dates = train_best_data.index.intersection(test_best_data.index)
test_best_data = test_best_data.drop(index=common_dates)

X_train = train_best_data[BEST_FEATURES]
y_train = train_best_data[TARGET_COL]

X_test = test_best_data[BEST_FEATURES]
y_test = test_best_data[TARGET_COL]

print(f"Treino: {X_train.index.min().date()} -> {X_train.index.max().date()} | {len(X_train)} obs")
print(f"Teste:  {X_test.index.min().date()} -> {X_test.index.max().date()} | {len(X_test)} obs")


In [ ]:
results = []
predictions = {}

for model_name, model in models.items():
    fitted_model = clone(model)
    fitted_model.fit(X_train, y_train)

    y_pred_train = fitted_model.predict(X_train)
    y_pred_test = fitted_model.predict(X_test)

    train_rmse, train_mae, train_wape = regression_metrics(y_train, y_pred_train)
    test_rmse, test_mae, test_wape = regression_metrics(y_test, y_pred_test)

    results.append({
        "model": model_name,
        "train_rmse": train_rmse,
        "train_mae": train_mae,
        "train_wape": train_wape,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_wape": test_wape,
        "rmse_increase_pct": relative_error_increase_pct(train_rmse, test_rmse),
        "mae_increase_pct": relative_error_increase_pct(train_mae, test_mae),
        "wape_increase_pct": relative_error_increase_pct(train_wape, test_wape),
        "Overfitting?": has_overfitting(
            [train_rmse, train_mae, train_wape],
            [test_rmse, test_mae, test_wape],
        ),
    })

    predictions[model_name] = pd.DataFrame(
        {"y_true": y_test, "y_pred": y_pred_test},
        index=y_test.index,
    )

best_features_metrics_df = pd.DataFrame(results).sort_values("test_rmse").reset_index(drop=True)
best_features_metrics_df = rank_models_by_average_test_rank(best_features_metrics_df)
display(best_features_metrics_df)


In [ ]:
def plot_top_best_model_metrics(best_features_metrics_df, top_n=1):
    top_models = best_features_metrics_df.sort_values("test_rmse").head(top_n).copy()

    metric_labels = ["RMSE", "MAE", "WAPE (%)"]
    train_cols = ["train_rmse", "train_mae", "train_wape"]
    test_cols = ["test_rmse", "test_mae", "test_wape"]

    for _, row in top_models.iterrows():
        model_name = row["model"]
        overfitting = row["Overfitting?"]

        train_values = [row[col] for col in train_cols]
        test_values = [row[col] for col in test_cols]

        x = np.arange(len(metric_labels))
        width = 0.35

        fig, ax = plt.subplots(figsize=(11, 6.5))

        bars_train = ax.bar(
            x - width / 2,
            train_values,
            width,
            label="Treino fitted",
            color=colors["Treino"],
            edgecolor="white",
            linewidth=0.7,
        )

        bars_test = ax.bar(
            x + width / 2,
            test_values,
            width,
            label="Teste forecast",
            color=colors["Teste"],
            edgecolor="white",
            linewidth=0.7,
        )

        ymax = max(max(train_values), max(test_values))
        ax.set_ylim(0, ymax * 1.22)

        fig.suptitle(
            f"{model_name}: treino vs teste",
            fontsize=14,
            fontweight="bold",
            y=0.97,
        )

        fig.legend(
            handles=[bars_train, bars_test],
            labels=["Treino fitted", "Teste forecast"],
            loc="upper center",
            bbox_to_anchor=(0.5, 0.91),
            ncol=2,
            frameon=False,
        )

        ax.set_ylabel("Valor", fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(metric_labels, fontsize=10)
        ax.tick_params(axis="y", labelsize=10)
        ax.grid(axis="y", alpha=0.25)
        ax.set_axisbelow(True)

        for bars in [bars_train, bars_test]:
            labels = [f"{bar.get_height():.2f}" for bar in bars]
            ax.bar_label(
                bars,
                labels=labels,
                padding=3,
                fontsize=9,
                fontweight="bold",
            )

        fig.subplots_adjust(top=0.80)
        plt.show()


plot_top_best_model_metrics(best_features_metrics_df, top_n=1)



---

## **Optimizing the Best Model Achieved (Best Features)**

In [ ]:
X_train = train_best_data[BEST_FEATURES]
y_train = train_best_data[TARGET_COL]

# Split temporal dentro do treino
split_idx = int(len(X_train) * 0.8)
X_tr, X_val = X_train.iloc[:split_idx], X_train.iloc[split_idx:]
y_tr, y_val = y_train.iloc[:split_idx], y_train.iloc[split_idx:]

def objective(trial):
    alpha = trial.suggest_float("alpha", 1e-2, 10.0, log=True)

    model = Ridge(
        alpha=alpha,
        fit_intercept=True,
        random_state=RANDOM_STATE,
    )
    model.fit(X_tr, y_tr)

    y_pred_val = model.predict(X_val)
    val_rmse, val_mae, val_wape = regression_metrics(y_val, y_pred_val)

    trial.set_user_attr("val_rmse", val_rmse)
    trial.set_user_attr("val_mae", val_mae)
    return val_wape

study_best_features = optuna.create_study(direction="minimize")
study_best_features.optimize(objective, n_trials=1000, show_progress_bar=False)

print("Best params:", study_best_features.best_params)
print("Best validation WAPE (%):", study_best_features.best_value)
print("Best validation RMSE:", study_best_features.best_trial.user_attrs["val_rmse"])
print("Best validation MAE:", study_best_features.best_trial.user_attrs["val_mae"])

best_ridge = Ridge(
    alpha=study_best_features.best_params["alpha"],
    fit_intercept=True,
    random_state=RANDOM_STATE,
)
best_ridge.fit(X_train, y_train)

y_pred_train = best_ridge.predict(X_train)
y_pred_test = best_ridge.predict(X_test)

train_rmse, train_mae, train_wape = regression_metrics(y_train, y_pred_train)
test_rmse, test_mae, test_wape = regression_metrics(y_test, y_pred_test)

print("Train RMSE:", train_rmse)
print("Train MAE:", train_mae)
print("Train WAPE (%):", train_wape)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test WAPE (%):", test_wape)
print("Overfitting?", has_overfitting(
    [train_rmse, train_mae, train_wape],
    [test_rmse, test_mae, test_wape]
))



# **Implementing the Best Model (Best Features) with the Best Parameters**

In [ ]:
BEST_ALPHA = study_best_features.best_params["alpha"] if "study_best_features" in globals() else 0.2097820888688671 # valor obtido na ultima execução

best_ridge = Ridge(
    alpha=BEST_ALPHA,
    fit_intercept=True,
    random_state=RANDOM_STATE,
)

best_ridge.fit(X_train, y_train)

y_pred_train = best_ridge.predict(X_train)
y_pred_test = best_ridge.predict(X_test)

train_rmse, train_mae, train_wape = regression_metrics(y_train, y_pred_train)
test_rmse, test_mae, test_wape = regression_metrics(y_test, y_pred_test)

print("Train RMSE:", train_rmse)
print("Train MAE:", train_mae)
print("Train WAPE (%):", train_wape)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test WAPE (%):", test_wape)
print("Overfitting?", has_overfitting(
    [train_rmse, train_mae, train_wape],
    [test_rmse, test_mae, test_wape]
))

coef_df = pd.DataFrame({
    "variavel": X_train.columns,
    "coeficiente": best_ridge.coef_,
    "abs_coef": abs(best_ridge.coef_),
}).sort_values("abs_coef", ascending=False).reset_index(drop=True)

print("\nIntercepto:")
print(best_ridge.intercept_)

print("\nCoeficientes ordenados por impacto absoluto:")
display(coef_df[["variavel", "coeficiente"]])

In [ ]:
optimized_ridge_metrics_df = pd.DataFrame([{
    "model": "Ridge",
    "train_rmse": train_rmse,
    "train_mae": train_mae,
    "train_wape": train_wape,
    "test_rmse": test_rmse,
    "test_mae": test_mae,
    "test_wape": test_wape,
    "Overfitting?": has_overfitting(
        [train_rmse, train_mae, train_wape],
        [test_rmse, test_mae, test_wape]
    )
}])

plot_top_best_model_metrics(optimized_ridge_metrics_df, top_n=1)

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Séries com índice temporal
fitted_train = pd.Series(y_pred_train, index=y_train.index, name="Ridge - Fitted Train")
forecast_test = pd.Series(y_pred_test, index=y_test.index, name="Ridge - Forecast Test")

fig = go.Figure()

# Valores reais no treino
fig.add_trace(go.Scatter(
    x=y_train.index,
    y=y_train,
    mode="lines",
    name="Real Train",
    line=dict(color="#8E9091", width=2)
))

# Fitted values no treino
fig.add_trace(go.Scatter(
    x=fitted_train.index,
    y=fitted_train,
    mode="lines",
    name="Ridge Fitted Train",
    line=dict(color="#3634bc", width=2)
))

# Valores reais no teste
fig.add_trace(go.Scatter(
    x=y_test.index,
    y=y_test,
    mode="lines",
    name="Real Test",
    line=dict(color="#b0b0b0", width=2, dash="dot")
))

# Previsão no teste
fig.add_trace(go.Scatter(
    x=forecast_test.index,
    y=forecast_test,
    mode="lines",
    name="Ridge Forecast Test",
    line=dict(color="#d62728", width=2)
))

fig.update_layout(
    title="Ridge: Fitted Values no Treino e Previsão no Teste",
    xaxis_title="Data",
    yaxis_title="Inflation Target",
    template="plotly_white",
    font=dict(family="Times New Roman", size=12),
    height=550,
    hovermode="x unified"
)

fig.show()

---

## **Optimizing the Best Model Achieved (Theoretical Features)**

In [ ]:
import optuna
from lightgbm import LGBMRegressor
from lightgbm.callback import early_stopping

# Mesmo dataset do modelo teórico com lag=10
X_train, y_train, X_test, y_test = get_lag_xy(
    lag=10,
    include_target_lag=False,   # manter igual ao bloco onde o LGBM lag=10 foi testado
)

# Split temporal dentro do treino
split_idx = int(len(X_train) * 0.8)
X_tr, X_val = X_train.iloc[:split_idx], X_train.iloc[split_idx:]
y_tr, y_val = y_train.iloc[:split_idx], y_train.iloc[split_idx:]

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 80),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 40),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": RANDOM_STATE,
        "verbose": -1,
    }

    model = LGBMRegressor(**params)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="l2",
        callbacks=[early_stopping(50, verbose=False)],
    )

    y_pred_val = model.predict(X_val)
    val_rmse, val_mae, val_wape = regression_metrics(y_val, y_pred_val)

    trial.set_user_attr("val_rmse", val_rmse)
    trial.set_user_attr("val_mae", val_mae)
    trial.set_user_attr("best_iteration", model.best_iteration_)
    return val_wape

study_theoretical = optuna.create_study(direction="minimize")
study_theoretical.optimize(objective, n_trials=1000, show_progress_bar=False)

print("Best params:", study_theoretical.best_params)
print("Best validation WAPE (%):", study_theoretical.best_value)
print("Best validation RMSE:", study_theoretical.best_trial.user_attrs["val_rmse"])
print("Best validation MAE:", study_theoretical.best_trial.user_attrs["val_mae"])
print("Best iteration:", study_theoretical.best_trial.user_attrs["best_iteration"])

best_params = study_theoretical.best_params.copy()
best_params.update({
    "random_state": RANDOM_STATE,
    "verbose": -1,
})

best_lgbm = LGBMRegressor(**best_params)
best_lgbm.fit(X_train, y_train)

y_pred_train = best_lgbm.predict(X_train)
y_pred_test = best_lgbm.predict(X_test)

train_rmse, train_mae, train_wape = regression_metrics(y_train, y_pred_train)
test_rmse, test_mae, test_wape = regression_metrics(y_test, y_pred_test)

print("Train RMSE:", train_rmse)
print("Train MAE:", train_mae)
print("Train WAPE (%):", train_wape)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test WAPE (%):", test_wape)
print("Overfitting?", has_overfitting(
    [train_rmse, train_mae, train_wape],
    [test_rmse, test_mae, test_wape]
))

# **Implementing the Best Model (Theoretical Features) with the Best Parameters**

In [ ]:
BEST_LGBM_PARAMS = study_theoretical.best_params if "study_theoretical" in globals() else {
    "n_estimators": 500,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "max_depth": 6,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.1,
}

best_lgbm = LGBMRegressor(
    **BEST_LGBM_PARAMS,
    random_state=RANDOM_STATE,
    verbose=-1,
)

best_lgbm.fit(X_train, y_train)

y_pred_train = best_lgbm.predict(X_train)
y_pred_test = best_lgbm.predict(X_test)

train_rmse, train_mae, train_wape = regression_metrics(y_train, y_pred_train)
test_rmse, test_mae, test_wape = regression_metrics(y_test, y_pred_test)

print("Train RMSE:", train_rmse)
print("Train MAE:", train_mae)
print("Train WAPE (%):", train_wape)
print("Test RMSE:", test_rmse)
print("Test MAE:", test_mae)
print("Test WAPE (%):", test_wape)
print("Overfitting?", has_overfitting(
    [train_rmse, train_mae, train_wape],
    [test_rmse, test_mae, test_wape]
))

importance_df = pd.DataFrame({
    "variavel": X_train.columns,
    "importancia": best_lgbm.feature_importances_,
}).sort_values("importancia", ascending=False).reset_index(drop=True)

print("\nMelhores hiperparâmetros:")
print(BEST_LGBM_PARAMS)

print("\nImportância das variáveis:")
display(importance_df)

---

## **Ploting models**

In [ ]:
from sklearn.base import clone
import plotly.graph_objects as go
from IPython.display import HTML, display

def build_model(variables, models=models):
    scores_train = {}
    scores_test = {}
    predictions_test = {}

    model_cols = [TARGET_COL] + variables
    train_data = df_train[model_cols].dropna().copy()
    test_data = df_test[model_cols].dropna().copy()

    common_dates = train_data.index.intersection(test_data.index)
    test_data = test_data.drop(index=common_dates)

    X_train = train_data[variables]
    y_train = train_data[TARGET_COL]

    X_test = test_data[variables]
    y_test = test_data[TARGET_COL]

    for model_name, clf in models.items():
        fitted_model = clone(clf)
        fitted_model.fit(X_train, y_train)

        pred_train = fitted_model.predict(X_train)
        pred_test = fitted_model.predict(X_test)

        scores_train[model_name] = regression_metrics(y_train, pred_train)
        scores_test[model_name] = regression_metrics(y_test, pred_test)
        predictions_test[model_name] = pred_test

    return scores_train, scores_test, predictions_test, models


def plot_predictions(y_true, predictions, models_to_plot=None, title="Previsoes vs Valores Reais"):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=y_true.index,
            y=y_true.values,
            mode="lines",
            name="Valores Reais",
            line=dict(color="#8E9091", width=2.5),
        )
    )

    if models_to_plot is None:
        models_to_plot = list(predictions.keys())

    for model_name in models_to_plot:
        line_color = "#1800ad" if len(models_to_plot) == 1 else None

        fig.add_trace(
            go.Scatter(
                x=y_true.index,
                y=predictions[model_name],
                mode="lines",
                name=model_name,
                line=dict(color=line_color, width=2),
            )
        )

    fig.update_layout(
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(family="Times New Roman", size=14),
        ),
        xaxis_title="Data",
        yaxis_title="Valor",
        template="plotly_white",
        hovermode="x unified",
        height=560,
        margin=dict(l=50, r=30, t=90, b=80),
        font=dict(family="Times New Roman", size=12),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
            font=dict(family="Times New Roman", size=12),
        ),
    )

    fig.update_xaxes(
        tickangle=-35,
        title_font=dict(family="Times New Roman", size=14),
        tickfont=dict(family="Times New Roman", size=12),
    )

    fig.update_yaxes(
        title_font=dict(family="Times New Roman", size=14),
        tickfont=dict(family="Times New Roman", size=12),
    )

    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

In [ ]:
# 1) Em nível
train, test, preds, _ = build_model(THEORETICAL_VARIABLES)
best_model_level = theoretical_metrics_df.iloc[0]["model"]

print(f"Best model in level: {best_model_level}")
plot_predictions(
    df_test[TARGET_COL],
    preds,
    models_to_plot=[best_model_level],
    title=f"Best model in level (Theoretical Features): {best_model_level}",
)

# 2) Desfasamento da variável alvo até lag 3

def performance_to_df(scores_train, scores_test):
    rows = []
    for model_name in scores_train.keys():
        rows.append({
            "model": model_name,
            "train_rmse": scores_train[model_name][0],
            "train_mae": scores_train[model_name][1],
            "train_wape": scores_train[model_name][2],
            "test_rmse": scores_test[model_name][0],
            "test_mae": scores_test[model_name][1],
            "test_wape": scores_test[model_name][2],
        })
    return pd.DataFrame(rows)


target_lags_1 = [f"{TARGET_COL}_lag_1"]
target_lags_2 = [f"{TARGET_COL}_lag_{i}" for i in range(1, 3)]
target_lags_3 = [f"{TARGET_COL}_lag_{i}" for i in range(1, 4)]

train1, test1, preds1, _ = build_model(target_lags_1)
train2, test2, preds2, _ = build_model(target_lags_2)
train3, test3, preds3, _ = build_model(target_lags_3)

perf1 = performance_to_df(train1, test1)
perf2 = performance_to_df(train2, test2)
perf3 = performance_to_df(train3, test3)

rank1 = rank_models_by_average_test_rank(perf1)
rank2 = rank_models_by_average_test_rank(perf2)
rank3 = rank_models_by_average_test_rank(perf3)

best_model_lag1 = rank1.iloc[0]["model"]
best_model_lag2 = rank2.iloc[0]["model"]
best_model_lag3 = rank3.iloc[0]["model"]

preds_target_lags = {
    f"Lag 1 ({best_model_lag1})": preds1[best_model_lag1],
    f"Lag 2 ({best_model_lag2})": preds2[best_model_lag2],
    f"Lag 3 ({best_model_lag3})": preds3[best_model_lag3],
}

print(f"Best model for lag 1: {best_model_lag1}")
print(f"Best model for lag 2: {best_model_lag2}")
print(f"Best model for lag 3: {best_model_lag3}")

plot_predictions(
    df_test[TARGET_COL],
    preds_target_lags,
    models_to_plot=[
        f"Lag 1 ({best_model_lag1})",
        f"Lag 2 ({best_model_lag2})",
        f"Lag 3 ({best_model_lag3})",
    ],
    title="Valores reais vs previsões com lags até 3",
)


# 3) Melhor número de lags teóricas vs melhor número de lags computacionais


best_lag_theoretical = int(lag_performance_df_theoretical.loc[0, "lag"])
best_model_theoretical = lag_performance_df_theoretical.loc[0, "model"]

lags_theoretical = [
    f"{key}_lag_{v}"
    for key, values in var_lags.items()
    if key in THEORETICAL_VARIABLES
    for v in values
    if v <= best_lag_theoretical
]

X_train_theoretical = df_train[THEORETICAL_VARIABLES + lags_theoretical].dropna()
y_train_theoretical = df_train[TARGET_COL].iloc[best_lag_theoretical:]

X_test_theoretical = df_test[THEORETICAL_VARIABLES + lags_theoretical]
pred_theoretical = clone(models[best_model_theoretical]).fit(
    X_train_theoretical, y_train_theoretical
).predict(X_test_theoretical)


best_lag_computacional = int(lag_performance_df_computacional.loc[0, "lag"])
best_model_computacional = lag_performance_df_computacional.loc[0, "model"]

lags_computacional = [
    f"{key}_lag_{v}"
    for key, values in var_lags.items()
    if key in COMPUTACIONAL_VARIABLES
    for v in values
    if v <= best_lag_computacional
]

target_lags = ["inflation_target_lag_1"]

X_train_computacional = df_train[COMPUTACIONAL_VARIABLES + lags_computacional + target_lags].dropna()
y_train_computacional = df_train[TARGET_COL].iloc[best_lag_computacional:]

X_test_computacional = df_test[COMPUTACIONAL_VARIABLES + lags_computacional + target_lags]
pred_computacional = clone(models[best_model_computacional]).fit(
    X_train_computacional, y_train_computacional
).predict(X_test_computacional)


plot_predictions(
    df_test[TARGET_COL],
    {
        f"Teóricas | {best_model_theoretical} | lag {best_lag_theoretical}": pred_theoretical,
        f"Computacionais | {best_model_computacional} | lag {best_lag_computacional}": pred_computacional,
    },
    title="Valores reais vs melhor modelo teórico e melhor modelo computacional",
)

# 4) Best features

model_cols = [TARGET_COL] + BEST_FEATURES

train_best_data = df_train[model_cols].dropna().copy()
test_best_data = df_test[model_cols].dropna().copy()

common_dates = train_best_data.index.intersection(test_best_data.index)
test_best_data = test_best_data.drop(index=common_dates)

X_train = train_best_data[BEST_FEATURES]
y_train = train_best_data[TARGET_COL]

X_test = test_best_data[BEST_FEATURES]
y_test_local = test_best_data[TARGET_COL]

best_alpha = study_best_features.best_params["alpha"] if "study_best_features" in globals() else BEST_ALPHA

best_model_tuned = Ridge(
    alpha=best_alpha,
    fit_intercept=True,
    random_state=RANDOM_STATE,
)

best_model_tuned.fit(X_train, y_train)
y_pred_tuned = best_model_tuned.predict(X_test)

preds_tuned = {
    f"Ridge Tuned": y_pred_tuned
}

plot_predictions(
    y_true=y_test_local,
    predictions=preds_tuned,
    models_to_plot=list(preds_tuned.keys()),
    title=f"Melhor Modelo (Feature Importance): Ridge",
)


train, test, preds, _ = build_model(BEST_FEATURES)

all_models = list(best_features_metrics_df["model"])

plot_predictions(
    df_test[TARGET_COL],
    preds,
    models_to_plot=all_models,
    title="Comparação de todos os modelos com as melhores features",
)
